In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/computer_vision
!find "/content/drive/Shareddrives/Computer Vision/generation_output" -mindepth 1 -maxdepth 1 -exec cp -r -t /content/computer_vision/ {} +


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import json
import os
import sys

# --- CONFIGURATION ---
# Check these paths carefully!
MODEL_PATH = '/content/drive/Shareddrives/Computer Vision/faces_model_checkpoints/resnet18_faces_best_32.pt'
LABEL_MAP_PATH = '/content/drive/Shareddrives/Computer Vision/faces_model_checkpoints/label_map.json'
TEST_DIR = '/content/computer_vision'


# Parameters
IMG_SIZE = 32     # Updated to 32 as requested
NUM_CLASSES = 10

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Configuration set. Using device: {device}")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")

Configuration set. Using device: cuda
Image Size: 32x32


In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import os
import json
from PIL import Image

def load_label_map(path):
    """Reads the JSON label map and converts keys to integers."""
    if not os.path.exists(path):
        print(f"Error: Label map not found at {path}")
        return {}

    with open(path, 'r') as f:
        mapping = json.load(f)
    return {int(k): v for k, v in mapping.items()}

def get_model(model_path, num_classes, device):
    """Loads ResNet18 and applies your specific weights."""
    print(f"Loading model from: {model_path}")

    # 1. Initialize architecture
    model = models.resnet18(pretrained=False)


    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)


    # ---------------------------------------------------------

    # 2. Modify final layer for num_classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    # 3. Load Weights
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found at {model_path}")

    try:
        # Load checkpoint
        checkpoint = torch.load(model_path, map_location=device)

        # Handle different saving formats
        if isinstance(checkpoint, dict):
            if 'model_state' in checkpoint:
                print("Found 'model_state' key. Loading weights...")
                model.load_state_dict(checkpoint['model_state'])
            elif 'model_state_dict' in checkpoint:
                print("Found 'model_state_dict' key. Loading weights...")
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                print("Assuming dictionary is the state dict...")
                model.load_state_dict(checkpoint)
        else:
            print("Checkpoint is a full model object.")
            model = checkpoint

    except Exception as e:
        print(f"Error loading weights: {e}")
        return None

    model = model.to(device)
    model.eval()
    return model

def predict_image(image_path, model, transform, device):
    """Reads an image, transforms it, and predicts the class index."""
    try:
        image = Image.open(image_path).convert('RGB')
        image_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(image_tensor)
            _, preds = torch.max(outputs, 1)

        return preds.item()
    except Exception as e:
        return None

In [ ]:
# 1. Define Transforms
# Important: Resize must match IMG_SIZE (32)
data_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. Load Resources
idx_to_class = load_label_map(LABEL_MAP_PATH)
model = get_model(MODEL_PATH, NUM_CLASSES, device)

if model is None or not idx_to_class:
    print("Stopping execution due to missing model or label map.")
else:
    # 3. Initialize Counters
    correct_total = 0
    total_images = 0

    # Track stats per class
    # Maps class name (e.g., "0000") to count
    class_correct = {v: 0 for v in idx_to_class.values()}
    class_total = {v: 0 for v in idx_to_class.values()}

    print("\nStarting evaluation...")

    # 4. Iterate through folders 0000 -> 0009
    # We sort to ensure order 0000, 0001, etc.
    if os.path.exists(TEST_DIR):
        folders = sorted([f for f in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, f))])

        for class_folder_name in folders:
            # Only process if this folder is in our label map values (0000-0009)
            if class_folder_name not in class_correct:
                continue

            class_path = os.path.join(TEST_DIR, class_folder_name)
            print(f"Processing folder: {class_folder_name}...", end=" ")

            folder_count = 0
            for img_file in os.listdir(class_path):
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.pt')):
                    img_path = os.path.join(class_path, img_file)

                    # Predict
                    pred_idx = predict_image(img_path, model, data_transforms, device)

                    if pred_idx is not None:
                        pred_label_name = idx_to_class.get(pred_idx, "Unknown")

                        # Update Stats
                        total_images += 1
                        class_total[class_folder_name] += 1
                        folder_count += 1

                        if pred_label_name == class_folder_name:
                            correct_total += 1
                            class_correct[class_folder_name] += 1

            print(f"Done ({folder_count} images)")

        # 5. Print Final Report
        print("\n" + "="*40)
        print(f"{'Class':<10} | {'Accuracy':<10} | {'Correct/Total'}")
        print("="*40)

        for class_name in sorted(class_total.keys()):
            total = class_total[class_name]
            correct = class_correct[class_name]
            acc = (correct / total * 100) if total > 0 else 0.0
            print(f"{class_name:<10} | {acc:<6.2f}%    | {correct}/{total}")

        print("-" * 40)
        overall_acc = (correct_total / total_images * 100) if total_images > 0 else 0
        print(f"OVERALL ACCURACY: {overall_acc:.2f}% ({correct_total}/{total_images})")

    else:
        print(f"Test directory not found: {TEST_DIR}")

Loading model from: /content/drive/Shareddrives/Computer Vision/faces_model_checkpoints/resnet18_faces_best_32.pt


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Found 'model_state' key. Loading weights...

Starting evaluation...
Processing folder: 0000... Done (5000 images)
Processing folder: 0001... Done (5000 images)
Processing folder: 0002... Done (5000 images)
Processing folder: 0003... Done (5000 images)
Processing folder: 0004... Done (5000 images)
Processing folder: 0005... Done (5000 images)
Processing folder: 0006... Done (5000 images)
Processing folder: 0007... Done (5000 images)
Processing folder: 0008... Done (5000 images)
Processing folder: 0009... Done (5000 images)

Class      | Accuracy   | Correct/Total
0000       | 4.48  %    | 224/5000
0001       | 6.46  %    | 323/5000
0002       | 1.32  %    | 66/5000
0003       | 44.46 %    | 2223/5000
0004       | 33.80 %    | 1690/5000
0005       | 5.96  %    | 298/5000
0006       | 37.64 %    | 1882/5000
0007       | 17.74 %    | 887/5000
0008       | 0.00  %    | 0/5000
0009       | 27.96 %    | 1398/5000
----------------------------------------
OVERALL ACCURACY: 17.98% (8991/50000)
